# Day 017 Project — Tool-Using Assistant

Extend the tool chatbot from the exercises with **at least one new tool** of your own design.

## Requirements

1. Keep the two built-in tools: `calculate` and `get_weather`
2. Add **at least one new tool** — ideas below
3. Add the new tool to `TOOL_REGISTRY` and to `TOOLS`
4. Update `SYSTEM_PROMPT` to mention the new tool and when to use it
5. Run the scripted checks — they verify all built-in tools still work and your new tool is registered

## Tool Ideas

| Tool | Description |
|------|-------------|
| `convert_units(value, from_unit, to_unit)` | Convert between km/miles, kg/lbs, °C/°F |
| `count_words(text)` | Count words and characters in a string |
| `reverse_text(text)` | Reverse a string |
| `get_day_of_week(date)` | Return the day name for a YYYY-MM-DD date |
| `lookup_element(symbol)` | Return element name and atomic number |

Feel free to invent your own. Anything that benefits from code execution (exact computation, string manipulation, data lookup) over the model's internal knowledge is a good fit.

In [ ]:
import ollama

In [ ]:
import ast, operator

def calculate(expression: str) -> str:
    allowed = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant): return node.value
        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsafe expression: {ast.dump(node)}")
    result = _eval(ast.parse(expression, mode='eval').body)
    return str(result)

def get_weather(city: str) -> str:
    temperatures = {"london": "12°C", "tokyo": "22°C", "paris": "15°C",
                    "new york": "18°C", "sydney": "24°C"}
    temp = temperatures.get(city.lower(), "20°C")
    return f"The current temperature in {city} is {temp}."

# ── Add your new tool function(s) here ──────────────────────────────────
# def my_new_tool(arg1, arg2, ...):
#     pass


# ── Registry ─────────────────────────────────────────────────────────────
TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
    # "my_new_tool": my_new_tool,  # uncomment and rename when ready
}

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluate a mathematical expression and return the numeric result. "
                "Use this for any arithmetic. Pass the expression as a string."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '12 * 34'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Return the current temperature for a city. "
                "Use this when the user asks about current weather or temperature."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'London' or 'Tokyo'",
                    }
                },
                "required": ["city"],
            },
        },
    },
    # ── Add your new tool definition here ──────────────────────────────────
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "my_new_tool",
    #         "description": "...",
    #         "parameters": {...},
    #     },
    # },
]

In [ ]:
def has_tool_call(response): return bool(response["message"].get("tool_calls"))
def extract_tool_call(response):
    call = response["message"]["tool_calls"][0]
    return call["function"]["name"], call["function"]["arguments"]
def execute_tool(fn_name, fn_args, registry):
    if fn_name not in registry: raise KeyError(f"Unknown tool: {fn_name!r}")
    return registry[fn_name](**fn_args)
def append_tool_result(messages, assistant_msg, tool_output):
    return messages + [assistant_msg, {"role": "tool", "content": tool_output}]
def append_turn(history, user_text, assistant_text):
    return history + [{"role": "user", "content": user_text}, {"role": "assistant", "content": assistant_text}]
def tool_turn(history, user_input, tools, registry, model="llama3.2"):
    messages = history + [{"role": "user", "content": user_input}]
    response = ollama.chat(model=model, messages=messages, tools=tools)
    if has_tool_call(response):
        fn_name, fn_args = extract_tool_call(response)
        tool_output = execute_tool(fn_name, fn_args, registry)
        messages = append_tool_result(messages, response["message"], tool_output)
        response = ollama.chat(model=model, messages=messages, tools=tools)
    reply = response["message"]["content"]
    return reply, append_turn(history, user_input, reply)
def truncate_history(history, max_turns=10):
    if not history: return []
    if history[0]["role"] == "system": system, tail = [history[0]], history[1:]
    else: system, tail = [], history
    return system + tail[-(max_turns * 2):]
def reset_history(history):
    if history and history[0]["role"] == "system": return [history[0]]
    return []
def format_history(history):
    labels = {"user": "You", "assistant": "Bot"}
    return "\n".join(f"{labels.get(m['role'], m['role'].capitalize())}: {m['content']}"
                     for m in history if m["role"] != "system")

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a calculator and weather lookup. "
    "Use the calculate tool for any arithmetic. "
    "Use the get_weather tool when asked about current temperature or weather."
    # Add a sentence describing your new tool and when to use it:
    # " Use the my_new_tool tool when..."
)

def run_tool_chatbot(
    tools=TOOLS, registry=TOOL_REGISTRY,
    system_prompt=SYSTEM_PROMPT, model="llama3.2", max_turns=10,
) -> None:
    history = [{"role": "system", "content": system_prompt}]
    print("Tool chatbot ready. Commands: /quit  /reset  /history")
    print("-" * 50)
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!"); break
        if not user_input: continue
        if user_input.startswith("/"):
            if user_input == "/quit": print("Goodbye!"); break
            elif user_input == "/reset": history = reset_history(history); print("Bot: Conversation reset.")
            elif user_input == "/history": print(format_history(history) or "(no history yet)")
            else: print(f"Bot: Unknown command: {user_input}")
            continue
        reply, history = tool_turn(history, user_input, tools, registry, model)
        history = truncate_history(history, max_turns)
        print(f"Bot: {reply}")

## Checks

Run these to verify your extended chatbot is working correctly.

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: at least 3 tools registered (2 built-in + 1 new)
    try:
        assert len(TOOL_REGISTRY) >= 3, f"Expected at least 3 tools, got {len(TOOL_REGISTRY)}"
        assert len(TOOLS) >= 3, f"Expected at least 3 TOOLS definitions, got {len(TOOLS)}"
        passed += 1; print("✅ Check 1: at least 3 tools in registry and TOOLS list")
    except Exception as e:
        print(f"❌ Check 1: tool count — {e}")

    # Check 2: calculate still works
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        reply, _ = tool_turn([], "What is 2847 * 193?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        assert "549471" in reply, f"expected '549471' in reply, got {reply!r}"
        passed += 1; print("✅ Check 2: calculate tool returns correct result")
    except Exception as e:
        sys.stdout = old
        print(f"❌ Check 2: calculate tool — {e}")

    # Check 3: get_weather still works
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        reply2, _ = tool_turn([], "What is the weather in London?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        assert "12°C" in reply2 or "london" in reply2.lower(), \
            f"expected temperature in reply, got {reply2!r}"
        passed += 1; print("✅ Check 3: get_weather tool returns temperature")
    except Exception as e:
        sys.stdout = old
        print(f"❌ Check 3: get_weather tool — {e}")

    # Check 4: new tool function is callable via execute_tool
    try:
        extra_tools = [t for t in TOOL_REGISTRY if t not in ("calculate", "get_weather")]
        assert len(extra_tools) >= 1, "No new tool found in TOOL_REGISTRY"
        new_name = extra_tools[0]
        # Just verify it's callable (the function exists and is in the registry)
        assert callable(TOOL_REGISTRY[new_name]), f"{new_name} is not callable"
        passed += 1; print(f"✅ Check 4: new tool '{new_name}' is registered and callable")
    except Exception as e:
        print(f"❌ Check 4: new tool registration — {e}")

    # Check 5: new tool has a matching TOOLS definition
    try:
        extra_tools = [t for t in TOOL_REGISTRY if t not in ("calculate", "get_weather")]
        assert extra_tools, "Check 4 must pass first"
        new_name = extra_tools[0]
        tool_names_in_defs = [t["function"]["name"] for t in TOOLS]
        assert new_name in tool_names_in_defs, \
            f"'{new_name}' is in TOOL_REGISTRY but not in TOOLS definitions list"
        passed += 1; print(f"✅ Check 5: '{new_name}' has a matching TOOLS definition")
    except Exception as e:
        print(f"❌ Check 5: TOOLS definition — {e}")

    if passed == total:
        print("🎉 Project complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Bonus Challenges

- Add **two or more new tools** — e.g., a unit converter + a string reverser
- Chain two tool calls in one turn: ask the chatbot something that requires both temperature conversion and weather lookup
- Write a separate function `list_tools(registry)` that prints the names and docstrings of all registered tools
- Add a `/tools` slash command to the chatbot that calls `list_tools`